In [39]:
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

In [20]:
# ----------------------
# CONFIGURATION SECTION
# ----------------------

# --- Years ---
CURRENT_YEAR = 2026
PRIOR_YEAR = 2025

# --- Mode toggle ---
# False = build levy list fresh from current secured roll
# True  = seed levy list from prior-year final levy (analysis workbook style)
USE_PRIOR_YEAR_LEVY = False

# --- Parcel tax constants ---
PARCEL_TAX_AMOUNT = 59.00
TAX_CODE = "64504"

# --- Mammoth USD TRAs (from Board resolution Exhibit A) ---
MAMMOTH_TRAS = [
    "010-000", "010-001", "010-002", "010-003", "010-004", "010-005", "010-006", "010-007",
    "010-008", "010-009", "010-010", "010-011", "010-012", "010-013", "010-014", "010-015",
    "059-000", "059-005", "059-007", "059-012", "059-018",
]

# --- File locations (TEMP: single analysis workbook) ---
ANALYSIS_WORKBOOK_NAME = "601_Secured_2025 - Analysis.xlsx"
ANALYSIS_WORKBOOK_PATH = Path.home() / "Desktop" / ANALYSIS_WORKBOOK_NAME

# --- Sheet names inside analysis workbook ---
SECURED_CURRENT_SHEET = "AGENCYCDCURRSEC_TR601"
NONTAXABLE_CURRENT_SHEET = "2025 Nontaxable"
SENIOR_PRIOR_SHEET = "Prior Year Senior Exemptions"
PRIOR_FINAL_LEVY_SHEET = "FINAL 2025-26 Tax Levy File"   # used only if USE_PRIOR_YEAR_LEVY = True


### Load in File

In [23]:
# =========================
# FILE IMPORT SECTION
# =========================

# List available sheets (sanity check)
xls = pd.ExcelFile(ANALYSIS_WORKBOOK_PATH)
print("Sheets in analysis workbook:")
for s in xls.sheet_names:
    print(" -", s)

# Load required sheets
secured_df = pd.read_excel(ANALYSIS_WORKBOOK_PATH, sheet_name=SECURED_CURRENT_SHEET)
nontaxable_df = pd.read_excel(ANALYSIS_WORKBOOK_PATH, sheet_name=NONTAXABLE_CURRENT_SHEET)
senior_df = pd.read_excel(ANALYSIS_WORKBOOK_PATH, sheet_name=SENIOR_PRIOR_SHEET)

# Load prior-year levy only if cumulative mode is enabled
prior_levy_df = (
    pd.read_excel(ANALYSIS_WORKBOOK_PATH, sheet_name=PRIOR_FINAL_LEVY_SHEET, header=None)
    if USE_PRIOR_YEAR_LEVY
    else None
)

print("\n✔ Input files loaded successfully. Ready to build levy base.")


Sheets in analysis workbook:
 - Instructions
 - AGENCYCDCURRSEC_TR601
 - 2024-25 Tax Levy File
 - PREVIOUS Nontaxable APNs
 - 2025 Nontaxable
 - Prior Year Senior Exemptions
 - New Senior Exemption
 - VSE that Changed
 - 2025 VSE
 - MammothTRAs2025
 - NEW APNs 2025
 - FINAL 2025-26 Tax Levy File

✔ Input files loaded successfully. Ready to build levy base.


### Verify TRAs (Work In Progress)

In [11]:
# # =========================
# # SETUP STEP 2: Identify key columns (APN, TRA, Owner)
# # =========================

# print("Total columns:", len(SECURED_COLS))
# print("\nAll column names:")
# for c in SECURED_COLS:
#     print(" -", c)

# # These are our expected names (edit if your file uses different headers)
# APN_COL = "APN"
# TRA_COL = "TRA"
# OWNER_COL = "Owner"

# missing = [c for c in [APN_COL, TRA_COL, OWNER_COL] if c not in SECURED_DF.columns]
# if missing:
#     raise KeyError(
#         f"Missing expected column(s): {missing}\n"
#         "Update APN_COL / TRA_COL / OWNER_COL to match your actual headers above."
#     )

# print("\nKey columns confirmed:")
# print("APN_COL  =", APN_COL)
# print("TRA_COL  =", TRA_COL)
# print("OWNER_COL=", OWNER_COL)

# print("\nSample APNs:")
# print(SECURED_DF[APN_COL].dropna().head(10).tolist())

# print("\nSample TRAs:")
# print(SECURED_DF[TRA_COL].dropna().head(10).tolist())

# print("\nSample Owners:")
# print(SECURED_DF[OWNER_COL].dropna().head(10).tolist())


### Filter to Mammoth TRAs to produce the "Universe"

In [27]:
# =========================
# STAGE 1: Mammoth parcels (fix TRA format, in-memory only)
# =========================

tra6 = secured_df["TRA"].astype("Int64").astype(str).str.zfill(6)
secured_df["TRA_norm"] = tra6.str[:3] + "-" + tra6.str[3:]

mammoth_df = secured_df[secured_df["TRA_norm"].isin(MAMMOTH_TRAS)].copy()

print("Mammoth parcel rows:", len(mammoth_df))
print(mammoth_df["TRA_norm"].value_counts())

Mammoth parcel rows: 11070
TRA_norm
010-006    7446
010-005    1158
010-008     559
010-001     524
059-007     518
010-004     316
059-012     217
010-009     132
059-005      94
059-000      57
010-007      29
010-000       9
010-002       9
010-014       1
010-015       1
Name: count, dtype: int64


In [32]:
# =========================
# STAGE 2A: Apply non-taxable exclusions
# =========================

base_df = mammoth_df.copy()

# Ensure 12-digit APN on base
base_df["APN_12"] = (
    base_df["APN"].astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# Identify APN column in non-taxable list
NONTAX_APN_COL = next(
    c for c in nontaxable_df.columns
    if "APN" in c.upper() or "NUMBER" in c.upper()
)

# Normalize non-taxable APNs
nontaxable_df["APN_12"] = (
    nontaxable_df[NONTAX_APN_COL].astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# Apply exclusion
nontax_set = set(nontaxable_df["APN_12"])
before = len(base_df)
after_nontax_df = base_df[~base_df["APN_12"].isin(nontax_set)].copy()

# Diagnostics
print("\n=== STAGE 2A: NON-TAXABLE EXCLUSIONS ===")
print("Parcels before:", before)
print("Removed (non-taxable):", before - len(after_nontax_df))
print("Remaining:", len(after_nontax_df))

# Optional review
unmatched_nontax = sorted(nontax_set - set(base_df["APN_12"]))
unmatched_nontax[:25]



=== STAGE 2A: NON-TAXABLE EXCLUSIONS ===
Parcels before: 11070
Removed (non-taxable): 45
Remaining: 11025


[]

In [41]:
# =========================
# STAGE 2B: Senior exemption review (review-only)
# =========================

# --- Choose base universe (toggle-aware) ---
base_df = (
    prior_levy_df.copy()
    if USE_PRIOR_YEAR_LEVY
    else after_nontax_df.copy()
)

# Ensure APN_12
base_df["APN_12"] = (
    base_df["APN"].astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# Ensure TRA_norm for context
if "TRA_norm" not in base_df.columns and "TRA" in base_df.columns:
    tra6 = base_df["TRA"].astype("Int64").astype(str).str.zfill(6)
    base_df["TRA_norm"] = tra6.str[:3] + "-" + tra6.str[3:]

# -------------------------
# Prepare senior review dataframe
# -------------------------
senior_review_df = senior_df.copy()

# Normalize senior APNs
senior_review_df["APN_12"] = (
    senior_review_df["APN"].astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# Join current Owner + TRA onto senior list
base_lookup = (
    base_df[["APN_12", "Owner", "TRA_norm"]]
    .drop_duplicates("APN_12")
)

senior_review_df = senior_review_df.merge(
    base_lookup, on="APN_12", how="left"
)

# -------------------------
# Owner normalization + scoring
# -------------------------
DROP_TOKENS = {"TRUST", "TRUSTEE", "ET", "AL", "ETAL", "AND", "THE"}
PUNCT = ",.;()[]{}'\"-_/"  # must match replacement length

def normalize_owner(s):
    if not s or str(s).lower() == "nan":
        return ""
    s = str(s).upper().translate(str.maketrans(PUNCT, " " * len(PUNCT)))
    return " ".join(t for t in s.split() if t not in DROP_TOKENS)

def match_score(a, b):
    a_clean = normalize_owner(a)
    b_clean = normalize_owner(b)

    if not a_clean and not b_clean:
        return 1.0
    if not a_clean or not b_clean:
        return 0.0

    tok_a = set(a_clean.split())
    tok_b = set(b_clean.split())

    jaccard = (
        len(tok_a & tok_b) / len(tok_a | tok_b)
        if tok_a | tok_b else 1.0
    )

    seq = SequenceMatcher(None, a_clean, b_clean).ratio()

    return 0.6 * jaccard + 0.4 * seq

# -------------------------
# Pick effective senior owner
# -------------------------
senior_review_df["Senior_Effective_Owner"] = (
    senior_review_df.get("Current Owner")
    .where(
        senior_review_df.get("Current Owner").notna(),
        senior_review_df.get("Master File Owner")
    )
)

# -------------------------
# Compute review flags
# -------------------------
REVIEW_THRESHOLD = 0.90

senior_review_df["MatchScore"] = senior_review_df.apply(
    lambda r: match_score(r["Senior_Effective_Owner"], r["Owner"]),
    axis=1
)

senior_review_df["Needs_Review"] = (
    senior_review_df["MatchScore"] < REVIEW_THRESHOLD
)

# -------------------------
# Diagnostics & displays
# -------------------------
print("\n=== STAGE 2B: SENIOR OWNER REVIEW ===")
print("Base universe:", "PRIOR YEAR LEVY" if USE_PRIOR_YEAR_LEVY else "NEW LIST")
print("Review threshold:", REVIEW_THRESHOLD)
print("Total seniors reviewed:", len(senior_review_df))
print("Flagged for review:", int(senior_review_df["Needs_Review"].sum()))

print("\n--- Lowest match scores (highest concern) ---")
display(
    senior_review_df.loc[senior_review_df["Needs_Review"]]
    .sort_values("MatchScore")
    .loc[:, ["APN_12", "Senior_Effective_Owner", "Owner", "TRA_norm", "MatchScore"]]
    .head(25)
)

print("\n--- Sample of rows flagged for review ---")
display(
    senior_review_df.loc[senior_review_df["Needs_Review"]]
    .loc[:, ["APN_12", "Senior_Effective_Owner", "Owner", "TRA_norm", "MatchScore"]]
    .head(25)
)

print("\n--- Sample of rows NOT flagged (clean matches) ---")
display(
    senior_review_df.loc[~senior_review_df["Needs_Review"]]
    .sort_values("MatchScore", ascending=False)
    .loc[:, ["APN_12", "Senior_Effective_Owner", "Owner", "MatchScore"]]
    .head(10)
)

# -------------------------
# Keep pipeline unchanged (review-only)
# -------------------------
after_senior_df = base_df.copy()


=== STAGE 2B: SENIOR OWNER REVIEW ===
Base universe: NEW LIST
Review threshold: 0.9
Total seniors reviewed: 54
Flagged for review: 10

--- Lowest match scores (highest concern) ---


,APN_12,Senior_Effective_Owner,Owner,TRA_norm,MatchScore
27,060170023000,KNOTT KATHRYN,FARIS LIVING TRUST 8-19-24,059-007,0.048485
13,062090011000,FORSTENZER FAMILY TRUST,AGEE REVOCABLE TRUST 4-28-99,059-012,0.061538
47,039060011000,WAHL DAVID C & MARGARET A,MURRAY TRUST 9-30-10,010-006,0.082051
25,040021007000,JASTRAB DOUG,DEL FANTE JAMES M. & KIM CHRISTINA,010-006,0.088889
42,035270009000,SAVAGE FAMILY TRUST 02-14-18,JIANG ATHENA Y. & KRAFT BENJAMIN,010-006,0.120755
31,060170014000,LUDVIK WILLIAM A & EILEEN,LUDVIK REVOCABLE TRUST 2-7-24,059-007,0.233333
44,039070010000,SHUGART LEE A & ANNA E,SHUGART LIVING TUST 4-9-92,010-006,0.271212
16,062120014000,GILBREATH FAMILY TRUST,GILBREATH FAMILY TRUST 12-17-82,059-005,0.552195
23,062130001000,HARZARD FAMILY TRUST 08-17-16,HAZARD FAMILY TRUST 8-17-16,059-005,0.638961
48,032130013000,WALTERS FAMILY TRUST 02-15-01,WALTERS FAMILY TRUST 2-15-01,010-001,0.791111



--- Sample of rows flagged for review ---


,APN_12,Senior_Effective_Owner,Owner,TRA_norm,MatchScore
13,062090011000,FORSTENZER FAMILY TRUST,AGEE REVOCABLE TRUST 4-28-99,059-012,0.061538
16,062120014000,GILBREATH FAMILY TRUST,GILBREATH FAMILY TRUST 12-17-82,059-005,0.552195
23,062130001000,HARZARD FAMILY TRUST 08-17-16,HAZARD FAMILY TRUST 8-17-16,059-005,0.638961
25,040021007000,JASTRAB DOUG,DEL FANTE JAMES M. & KIM CHRISTINA,010-006,0.088889
27,060170023000,KNOTT KATHRYN,FARIS LIVING TRUST 8-19-24,059-007,0.048485
31,060170014000,LUDVIK WILLIAM A & EILEEN,LUDVIK REVOCABLE TRUST 2-7-24,059-007,0.233333
42,035270009000,SAVAGE FAMILY TRUST 02-14-18,JIANG ATHENA Y. & KRAFT BENJAMIN,010-006,0.120755
44,039070010000,SHUGART LEE A & ANNA E,SHUGART LIVING TUST 4-9-92,010-006,0.271212
47,039060011000,WAHL DAVID C & MARGARET A,MURRAY TRUST 9-30-10,010-006,0.082051
48,032130013000,WALTERS FAMILY TRUST 02-15-01,WALTERS FAMILY TRUST 2-15-01,010-001,0.791111



--- Sample of rows NOT flagged (clean matches) ---


,APN_12,Senior_Effective_Owner,Owner,MatchScore
0,033290037000,BAGGETT L RICHARD,"BAGGETT, L. RICHARD",1.0
1,033134014000,BENES FAMILY TRUST DTD 022007,BENES FAMILY TRUST DTD 022007,1.0
29,062130010000,LAVAGNINO ROBERT S & SUSAN B,LAVAGNINO ROBERT S. & SUSAN B,1.0
30,039040011000,LEHOTSKY FAMILY TRUST 11-8-21,LEHOTSKY FAMILY TRUST 11-8-21,1.0
32,022462013000,LUTHI LIVING TRUST 08-17-17,LUTHI LIVING TRUST 08-17-17,1.0
33,033270062000,MAGID SURVIVORS TRUST,MAGID SURVIVORS TRUST,1.0
34,032051077000,MARCINKO JOHN J. & DOROTHY,MARCINKO JOHN J. & DOROTHY,1.0
35,033270063000,MC GIMSEY-PEATROSS TRUST 04/02/2010,MC GIMSEY-PEATROSS TRUST 04/02/2010,1.0
36,032030014000,MOBLEY FAMILY TRUST 8-24-98,MOBLEY FAMILY TRUST 8-24-98,1.0
37,033320015000,MOORE 2008 TRUST 08-21-08,MOORE 2008 TRUST 08-21-08,1.0


In [43]:
# =========================
# STAGE 3: Build final levy table (in-memory)
# =========================

levy_df = (
    after_senior_df.assign(
        APN=after_senior_df["APN"].astype(str).str.replace(r"\D", "", regex=True).str.zfill(12),
        Fee=PARCEL_TAX_AMOUNT,
        TaxCode=TAX_CODE,
    )[["APN", "Fee", "TaxCode"]]
    .drop_duplicates("APN")
    .sort_values("APN")
    .reset_index(drop=True)
)

print("Final levy rows:", len(levy_df))
levy_df.head(20)


Final levy rows: 11025


,APN,Fee,TaxCode
0,014220005000,59.0,64504
1,014220006000,59.0,64504
2,014220014000,59.0,64504
3,014220015000,59.0,64504
4,014220016000,59.0,64504
5,014220017000,59.0,64504
6,014220018000,59.0,64504
7,014220019000,59.0,64504
8,014220020000,59.0,64504
9,014240008000,59.0,64504


### QA

In [ ]:
analysis_path = Path.home() / "Desktop" / "601_Secured_2025 - Analysis.xlsx"

FINAL_SHEET_NAME = "FINAL 2025-26 Tax Levy File"

final_analysis_df = pd.read_excel(
    analysis_path,
    sheet_name=FINAL_SHEET_NAME,
    header=None
)

print("Loaded analysis final levy sheet")
print("Rows:", len(final_analysis_df))
final_analysis_df.head(10)


Loaded analysis final levy sheet
Rows: 11000


,0,1,2
0,2.237001e+10,59.0,64504.0
1,3.904000e+10,59.0,64504.0
2,4.004241e+10,59.0,64504.0
3,6.021001e+10,59.0,64504.0
4,1.422000e+10,59.0,64504.0
5,1.422001e+10,59.0,64504.0
6,1.422001e+10,59.0,64504.0
7,1.422002e+10,59.0,64504.0
8,1.422002e+10,59.0,64504.0
9,1.422002e+10,59.0,64504.0


In [ ]:
# --- Rebuild analysis APN_12 correctly from column 0 (floats) ---
analysis_apn12 = (
    final_analysis_df[0]
    .dropna()                 # IMPORTANT: do NOT turn blanks into 0
    .astype("Int64")          # removes the .0 cleanly
    .astype(str)
    .str.zfill(12)            # restores leading zeros like 014220...
)

# --- Our APN_12 (already correct, but normalize similarly) ---
our_apn12 = (
    levy_df["APN_12"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

analysis_set = set(analysis_apn12)
our_set = set(our_apn12)

missing_in_ours = sorted(analysis_set - our_set)
extra_in_ours = sorted(our_set - analysis_set)

print("\n=== FINAL LEVY COMPARISON (CLEAN) ===")
print("Analysis unique APNs:", len(analysis_set))
print("Our unique APNs:", len(our_set))
print("Missing in ours:", len(missing_in_ours))
print("Extra in ours:", len(extra_in_ours))

if missing_in_ours:
    print("\nExamples missing in ours (first 25):")
    print(missing_in_ours[:25])

if extra_in_ours:
    print("\nExamples extra in ours (first 25):")
    print(extra_in_ours[:25])


KeyError: 'APN_12'

In [ ]:
missing_in_ours = sorted(analysis_set - our_set)
extra_in_ours = sorted(our_set - analysis_set)

print("\n=== FINAL LEVY COMPARISON ===")
print("Missing in ours:", len(missing_in_ours))
print("Extra in ours:", len(extra_in_ours))

if missing_in_ours:
    print("\nExamples missing in ours (first 25):")
    print(missing_in_ours[:25])

if extra_in_ours:
    print("\nExamples extra in ours (first 25):")
    print(extra_in_ours[:25])



=== FINAL LEVY COMPARISON ===
Missing in ours: 10984
Extra in ours: 10941

Examples missing in ours (first 25):
['000000000000', '142200050000', '142200060000', '142200140000', '142200150000', '142200160000', '142200170000', '142200180000', '142200190000', '142200200000', '142400080000', '142400090000', '142500010000', '142500020000', '142500030000', '142500040000', '221500030000', '221500040000', '221500050000', '222310010000', '222310020000', '222310030000', '222310050000', '222310060000', '222310070000']

Examples extra in ours (first 25):
['014220005000', '014220006000', '014220014000', '014220015000', '014220016000', '014220017000', '014220018000', '014220019000', '014220020000', '014240008000', '014240009000', '014250001000', '014250002000', '014250003000', '014250004000', '022150003000', '022150004000', '022150005000', '022231001000', '022231002000', '022231003000', '022231005000', '022231006000', '022231007000', '022231012000']


In [ ]:
# Focus only on missing APNs that ARE in mammoth_df
missing_in_mammoth_set = set(missing_in_mammoth)

# Normalize exemption APN sets
nontax_apns = set(
    nontax_df["APN_str"]
    .astype(str)
    .str.replace(r"\D","", regex=True)
    .str.zfill(12)
)

senior_apns = set(
    senior_df["APN_str"]
    .astype(str)
    .str.replace(r"\D","", regex=True)
    .str.zfill(12)
)

print("Of the 36 missing Mammoth APNs:")
print("In non-taxable list:", len(missing_in_mammoth_set & nontax_apns))
print("In senior list:", len(missing_in_mammoth_set & senior_apns))

print("\nExamples in non-taxable:")
print(list(missing_in_mammoth_set & nontax_apns)[:10])

print("\nExamples in senior:")
print(list(missing_in_mammoth_set & senior_apns)[:10])


Of the 36 missing Mammoth APNs:
In non-taxable list: 30
In senior list: 6

Examples in non-taxable:
['062200010100', '905001026000', '062200006100', '062040003100', '062200007000', '905001021000', '905001007000', '905001025000', '062200008100', '026120003000']

Examples in senior:
['035212014000', '035270009000', '040021007000', '062090011000', '039060011000', '060170023000']


### Export to CSV

In [ ]:
# =========================
# EXPORT: Final 2025–26 Tax Levy CSV
# =========================

out_path = Path.home() / "Desktop" / "64504 Mammoth USD 01252026.csv"

levy_df.to_csv(
    out_path,
    index=False
)

print("Saved:", out_path)


Saved: /Users/ewilson/Desktop/64504 Mammoth USD 01252026.csv
